# Feature Engineering
Building model-ready features from cleaned Strava data.
Features: rolling load, ACWR, HR efficiency, temporal features, and Whoop biometrics.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/processed/activities_clean.csv", parse_dates=["date"])
df["date"] = df["date"].dt.tz_localize(None)  # strip timezone
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded {len(df)} runs")
df.head()

Loaded 509 runs


,id,date,name,distance_km,duration_sec,elevation_m,avg_hr,max_hr,avg_speed,pace_min_per_km,week,day_of_week,month
0,5159424797,2021-04-20 12:50:09,Lunch Run,3.0163,925,0.0,NaN,NaN,3.261,5.111118,2021-04-19,Tuesday,April
1,5207562011,2021-04-28 16:18:00,Afternoon Run,2.0068,594,0.0,NaN,NaN,3.378,4.933227,2021-04-26,Wednesday,April
2,6227335431,2021-11-08 00:05:57,Midnight run,6.4928,2041,32.2,NaN,NaN,3.181,5.239137,2021-11-08,Monday,November
3,6944184769,2022-04-07 12:31:22,Lunch Run,2.5614,706,0.0,NaN,NaN,3.628,4.593842,2022-04-04,Thursday,April
4,7022950636,2022-04-22 16:16:56,Afternoon Run,2.0087,478,0.0,NaN,NaN,4.202,3.966081,2022-04-18,Friday,April


In [4]:
df["day_of_week"] = df["date"].dt.dayofweek      # 0=Monday, 6=Sunday
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour
df["time_of_day"] = pd.cut(df["hour"],
                            bins=[-1, 6, 12, 17, 21, 24],
                            labels=["night", "morning", "afternoon", "evening", "late"])

print(df[["date", "hour", "time_of_day"]].head(10))

                 date  hour time_of_day
0 2021-04-20 12:50:09    12     morning
1 2021-04-28 16:18:00    16   afternoon
2 2021-11-08 00:05:57     0       night
3 2022-04-07 12:31:22    12     morning
4 2022-04-22 16:16:56    16   afternoon
5 2022-05-17 20:16:09    20     evening
6 2022-07-31 20:09:54    20     evening
7 2022-11-12 18:54:26    18     evening
8 2022-11-29 15:27:07    15   afternoon
9 2023-02-28 19:05:09    19     evening


In [3]:
df["days_since_last_run"] = df["date"].diff().dt.days.fillna(0)

print(df[["date", "days_since_last_run"]].head(10))

                 date  days_since_last_run
0 2021-04-20 12:50:09                  0.0
1 2021-04-28 16:18:00                  8.0
2 2021-11-08 00:05:57                193.0
3 2022-04-07 12:31:22                150.0
4 2022-04-22 16:16:56                 15.0
5 2022-05-17 20:16:09                 25.0
6 2022-07-31 20:09:54                 74.0
7 2022-11-12 18:54:26                103.0
8 2022-11-29 15:27:07                 16.0
9 2023-02-28 19:05:09                 91.0


In [5]:
# Set date as index for rolling calculations
df = df.set_index("date")

df["rolling_7d_km"] = df["distance_km"].rolling("7D").sum()
df["rolling_28d_km"] = df["distance_km"].rolling("28D").sum()
df["rolling_42d_km"] = df["distance_km"].rolling("42D").sum()

df = df.reset_index()

print(df[["date", "distance_km", "rolling_7d_km", "rolling_28d_km"]].tail(10))

                   date  distance_km  rolling_7d_km  rolling_28d_km
499 2026-03-22 09:03:13      26.2052        59.8519         97.4534
500 2026-03-26 18:23:14       7.5249        33.7301         93.2687
501 2026-03-29 10:59:51      21.4345        28.9594        114.7032
502 2026-04-04 10:00:08      31.3833        52.8178        146.0865
503 2026-04-08 20:02:32       6.3605        37.7438        126.5551
504 2026-04-12 09:02:42      10.6095        16.9700        137.1646
505 2026-04-14 18:48:18       8.7112        25.6812        112.2291
506 2026-04-14 20:31:01       3.5041        29.1853        115.7332
507 2026-04-19 09:10:20      42.5428        54.7581        132.0708
508 2026-05-05 16:55:11      11.0201        11.0201         82.7482


In [8]:
# Only calculate ACWR where chronic load is meaningful (>5km)
df_daily["acwr"] = np.where(
    df_daily["rolling_28d_km"] > 5,
    df_daily["rolling_7d_km"] / df_daily["rolling_28d_km"],
    np.nan
)

# Smooth it slightly
df_daily["acwr_smooth"] = df_daily["acwr"].rolling(7).mean()

# Only plot from mid 2023 when training became consistent
df_plot = df_daily[df_daily.index >= "2023-02-01"]

fig = px.line(df_plot, x=df_plot.index, y="acwr_smooth",
              title="Acute:Chronic Workload Ratio (from consistent training period)",
              labels={"acwr_smooth": "ACWR", "x": "Date"})
fig.add_hline(y=1.5, line_dash="dash", line_color="red",
              annotation_text="Injury risk threshold")
fig.add_hline(y=0.8, line_dash="dash", line_color="orange",
              annotation_text="Undertraining threshold")
fig.show()

In [9]:
# Filter to runs with valid HR data
df_hr = df[df["avg_hr"].notna()].copy()

# HR efficiency = speed (m/min) / heart rate
# Higher = more speed per heartbeat = better fitness
df_hr["speed_m_per_min"] = (df_hr["distance_km"] * 1000) / (df_hr["duration_sec"] / 60)
df_hr["hr_efficiency"] = df_hr["speed_m_per_min"] / df_hr["avg_hr"]

# Filter to easy runs only (avg_hr < 155) to remove intensity confounding
df_easy = df_hr[df_hr["avg_hr"] < 155].copy()
print(f"Easy runs with HR: {len(df_easy)}")

fig = px.scatter(df_easy, x="date", y="hr_efficiency",
                 trendline="lowess",
                 title="HR Efficiency Over Time (easy runs only)",
                 labels={"hr_efficiency": "HR Efficiency (m/min/bpm)", "date": "Date"})
fig.show()

Easy runs with HR: 226


In [10]:
# Remove outliers — anything above 1.8 is likely a sensor error
df_easy = df_easy[df_easy["hr_efficiency"] < 1.8].copy()
print(f"Clean easy runs: {len(df_easy)}")

Clean easy runs: 225


In [11]:
df_easy["month_year"] = df_easy["date"].dt.to_period("M").dt.start_time

monthly_eff = df_easy.groupby("month_year")["hr_efficiency"].mean().reset_index()

fig = px.line(monthly_eff, x="month_year", y="hr_efficiency",
              title="Monthly Average HR Efficiency (easy runs)",
              labels={"hr_efficiency": "HR Efficiency (m/min/bpm)", 
                      "month_year": "Month"})
fig.show()

In [12]:
# Merge rolling load features from df_daily back to df
df_daily_features = df_daily[["rolling_7d_km", "rolling_28d_km", 
                               "rolling_42d_km", "acwr"]].reset_index()
df_daily_features.columns = ["date", "rolling_7d_km", "rolling_28d_km", 
                              "rolling_42d_km", "acwr"]

# Round date to day for merging
df["date_day"] = df["date"].dt.floor("D")
df_daily_features["date"] = pd.to_datetime(df_daily_features["date"])

df = df.merge(df_daily_features, left_on="date_day", right_on="date", 
              how="left", suffixes=("", "_daily"))
df = df.drop(columns=["date_daily", "date_day"])

print(df.columns.tolist())
print(df.shape)

['date', 'id', 'name', 'distance_km', 'duration_sec', 'elevation_m', 'avg_hr', 'max_hr', 'avg_speed', 'pace_min_per_km', 'week', 'day_of_week', 'month', 'hour', 'time_of_day', 'days_since_last_run', 'rolling_7d_km', 'rolling_28d_km', 'rolling_42d_km', 'acwr', 'rolling_7d_km_daily', 'rolling_28d_km_daily', 'rolling_42d_km_daily', 'acwr_daily']
(509, 24)


In [13]:
df.to_csv("../data/processed/activities_features.csv", index=False)
print("Saved feature engineered data")

Saved feature engineered data
